In [ ]:
## ================================
## 0) Libraries & read metadata
## ================================
library(readr)
library(dplyr)
library(tidyr)
library(ggplot2)
library(stringr)
library(janitor)
library(viridis)   # colour-blind friendly palettes
library(GGally)
library(tidyverse)

In [ ]:
#metadata from Kevin big excel ath_merge
library(googlesheets4)
gs4_deauth()  # Use public, read-only access

In [ ]:
library(googlesheets4)
library(readr)
library(dplyr)
library(stringr)
library(janitor)

# ---- METADATA FROM GOOGLE SHEETS ----
 meta <- read_sheet(
    "https://docs.google.com/REDACTED",
  sheet = "everything",
  col_types = "c"   # read as character, we'll coerce later
) %>%
  mutate(across(everything(), ~ ifelse(. == "NA", NA, .))) %>%  # keep real NAs
  mutate(sample = str_squish(sample),
         tube_id = str_squish(tube_id))

In [ ]:
dim(meta)
glimpse(meta)

In [ ]:
# ---- EXPRESSION MATRIX ----
expr_raw <- read_csv("20250710_RNana_Seqtk_Filtered_vst_limma_w.DesignMat.csv", show_col_types = FALSE)

In [ ]:
dim(expr_raw)
head(expr_raw)

In [ ]:
clean_names <- function(x) {
  x %>%
    str_replace("_[Rr]edo.*$", "") %>%    # remove _redo, _redo1, _Redo2 etc.
    str_replace("_[Vv][0-9]+$", "") %>%   # remove _v1, _V2...
    str_replace("_[Rr][0-9]+$", "") %>%   # remove _r1, _R2...
    str_replace("_[0-9]+$", "")           # remove pure numeric suffixes like _1
}

In [ ]:
expr_clean <- expr_raw
colnames(expr_clean) <- colnames(expr_clean) %>% clean_names()


In [ ]:
colnames(expr_clean)[1:20]

In [ ]:
expr_ids <- colnames(expr_clean)[-1]

In [ ]:
length(expr_ids)   # should be 616
head(expr_ids)

In [ ]:
meta_clean <- meta %>%
  mutate(sample_clean = clean_names(sample))

In [ ]:
head(meta_clean$sample_clean)

In [ ]:
meta_616 <- meta_clean %>%
  filter(sample_clean %in% expr_ids) %>%
  distinct(sample_clean, .keep_all = TRUE)

In [ ]:
nrow(meta_616)
length(unique(meta_616$sample_clean))

In [ ]:
meta_final <- tibble(sample_clean = expr_ids) %>%
  left_join(meta_616, by = "sample_clean")

In [ ]:
dim(meta_final)                     # should be 616 × (metadata columns)
identical(meta_final$sample_clean, expr_ids)   # TRUE

In [ ]:
glimpse(meta_final)

In [ ]:
# basic checks
nrow(meta_final)                        # expect 616
length(unique(meta_final$sample))       # expect 616

In [ ]:
# define df and season_short
meta_final <- meta_final %>%
  mutate(
    season_short = str_replace(season, "^(\\d{4})-\\d{2}-(.*)$", "\\1-\\2")
  )

table(meta_final$season_short)

In [ ]:
library(dplyr)
library(stringr)
library(lubridate)

guess_col_type <- function(x) {
  # Treat empty strings and literal "NA" as missing
  x <- ifelse(x %in% c("", "NA", "NaN", "na", "n/a"), NA, x)
  
  # If everything is NA → leave as character
  if (all(is.na(x))) return(x)
  
  n_non_na <- sum(!is.na(x))
  
  ## ---- Try numeric ----
  num_try <- suppressWarnings(as.numeric(str_replace(x, ",", ".")))
  if (sum(!is.na(num_try)) >= 0.9 * n_non_na) {
    return(num_try)
  }
  
  ## ---- Try logical (TRUE/FALSE, 0/1, yes/no) ----
  logi_map <- function(v) {
    case_when(
      v %in% c("TRUE","True","T","1","yes","Yes","Y")   ~ TRUE,
      v %in% c("FALSE","False","F","0","no","No","N")   ~ FALSE,
      TRUE                                              ~ NA
    )
  }
  logi_try <- logi_map(x)
  if (sum(!is.na(logi_try)) >= 0.9 * n_non_na) {
    return(logi_try)
  }
  
  ## ---- Try Date (ymd or dmy) ----
  d1 <- suppressWarnings(ymd(x))
  d2 <- suppressWarnings(dmy(x))
  date_try <- ifelse(!is.na(d1), d1, d2)
  if (sum(!is.na(date_try)) >= 0.9 * n_non_na) {
    return(as.Date(date_try))
  }
  
  ## ---- Try POSIXct (datetime) ----
  dt1 <- suppressWarnings(ymd_hms(x, tz = "UTC"))
  dt2 <- suppressWarnings(dmy_hms(x, tz = "UTC"))
  dt_try <- ifelse(!is.na(dt1), dt1, dt2)
  if (sum(!is.na(dt_try)) >= 0.9 * n_non_na) {
    return(as.POSIXct(dt_try, origin = "1970-01-01", tz = "UTC"))
  }
  
  ## ---- Fallback: keep as character ----
  return(x)
}


In [ ]:
meta_final_typed <- meta_final %>%
  mutate(across(everything(), guess_col_type))

glimpse(meta_final_typed)


In [ ]:
write.csv(meta_final_typed, "251206_MetaFinal_Complete.csv")